In [56]:
import pandas as pd

noblocks = pd.read_csv("local_blocked_docs_noblocks.lite.csv").rename(columns={"score": "noblocks_score"})
blocks = pd.read_csv("local_blocked_docs_blocks.lite.csv").rename(columns={"score": "blocks_score"})
D = blocks.merge(noblocks, on=["doc_id", "method"])[["blocks_score", "noblocks_score", "doc_id", "method"]].drop_duplicates()

hardness = blocks.merge(noblocks, on=["doc_id", "method"])[["blocks_score", "noblocks_score", "doc_id", "method"]].drop_duplicates()
hardness = hardness[hardness["method"] == "loss"]
hardness["hardness"] = (hardness["blocks_score"] + hardness["noblocks_score"])/2
doc2loss = {k:v for k, v in zip(hardness["doc_id"],hardness["hardness"])}
D["delta"] = D["blocks_score"] - D["noblocks_score"]
D["hardness"] = D["doc_id"].apply(lambda x: doc2loss[x])

In [60]:
D[["method", "delta"]].groupby(["method"]).mean().reset_index()

,method,delta
0,loss,0.047521
1,min_k,0.111336
2,zlib,0.000108


In [59]:
D[D["method"] == "loss"]["noblocks_score"].mean()

2.3189389035104777

In [58]:
len(D)

3000

In [47]:
D[D["method"] == "min_k"].sort_values("delta", ascending=False).iloc[200:300]

,blocks_score,noblocks_score,doc_id,method,delta,hardness
54550,10.956936,8.840733,https://www.knau.org/2023-07-29/a-resident-of-...,min_k,2.116203,2.280461
47520,2.600494,0.494112,https://www.kfyrtv.com/prnewswire/2022/07/05/t...,min_k,2.106382,0.325137
54334,11.315129,9.218805,https://www.wlrn.org/npr-breaking-news/npr-bre...,min_k,2.096324,3.239529
53346,2.334543,0.239380,https://www.kait8.com/prnewswire/2022/09/16/re...,min_k,2.095163,0.300323
48130,2.207165,0.119728,https://www.kwch.com/prnewswire/2022/07/26/phi...,min_k,2.087436,0.274717
...,...,...,...,...,...,...
54268,3.303002,1.569367,https://www.1011now.com/prnewswire/2022/07/19/...,min_k,1.733635,0.603745
54154,3.125438,1.399507,https://www.wlbt.com/prnewswire/2022/07/18/com...,min_k,1.725930,0.508631
45206,9.262396,7.537567,https://www.washingtonpost.com/world/2022/08/1...,min_k,1.724828,2.243412
40964,15.037637,13.312889,https://fox59.com/news/national-world/ap-inter...,min_k,1.724749,5.106186


In [48]:
import numpy as np
#Z = D[D["method"] == "loss"]
#for _, z in Z[Z["doc_id"].apply(lambda x: "/local/" in x)].sort_values("delta", ascending=False)[["doc_id", "delta"]].iterrows():
#    print(z['doc_id'], z['delta'])

In [55]:
Z = D[D["method"] == "min_k"]
Z = Z[Z["doc_id"].apply(lambda x: "/local/" in x)].copy()
Z["delta"].mean()

0.2416322858609822